# Dado desbalanceado: a acurácia mente e as quatro alavancas

Neste notebook o evento raro é 3% da base: quem cancela, quem fica doente, quem frauda. E é sobre esse 3%
que o problema existe.

A primeira coisa que o desbalanceamento quebra é a **métrica**. Um modelo que responde "não" para todo
mundo acerta 97% das vezes, e não serve para nada. A partir daí, existem quatro alavancas, e o notebook
mede cada uma:

1. **métrica**: parar de olhar acurácia e olhar precisão e revocação, e a curva de precisão-revocação;
2. **peso**: `class_weight="balanced"`, que pesa o erro da classe rara no treino sem inventar linha;
3. **reamostragem**: SMOTE (cria linha sintética da minoria) e subamostragem (joga fora linha da maioria),
   com a regra de que isso acontece **só no treino**;
4. **limiar**: não mexer no dado nem no modelo, e sim no ponto de corte da decisão.

E a armadilha que junta tudo: reamostrar antes de dividir.


In [ ]:
%pip install imbalanced-learn==0.14.2


In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as PipelineDesbalanceado
from imblearn.under_sampling import RandomUnderSampler
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
                             f1_score, precision_recall_curve, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", message=".*will be removed.*")
sns.set_theme(context="notebook", style="whitegrid", palette="colorblind",
              rc={"figure.figsize": (9, 4)})

RANDOM_STATE = 42
ALVO = "cancelou"

bruta = make_classification(n_samples=4000, n_features=12, n_informative=5, n_redundant=0,
                            weights=[0.97, 0.03], flip_y=0.01, class_sep=1.0,
                            random_state=RANDOM_STATE)
X = pd.DataFrame(bruta[0], columns=[f"variavel_{i:02d}" for i in range(1, 13)])
y = pd.Series(bruta[1], name=ALVO)

print(f"tabela: {X.shape[0]} linhas x {X.shape[1]} colunas")
print(f"alvo: {y.mean():.2%} de positivos ({int(y.sum())} casos em {len(y)})")
print()
print("o problema deste notebook é o que fazer com esses", int(y.sum()), "casos")

sns.countplot(y=y, color=sns.color_palette("colorblind")[2])
plt.title(f"o alvo: {y.mean():.1%} de positivos")
plt.yscale("log")
plt.show()


## A acurácia mente

O jeito mais rápido de ver isso é treinar um modelo que não aprende nada: responder sempre a classe
majoritária. Se a acurácia dele é alta, então a acurácia não serve para julgar nada aqui.


In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)


def mede(classificador, X_dados, y_alvo, nome):
    """Mede a acuracia e as metricas que olham o evento raro, na mesma validacao cruzada."""
    resultado = cross_validate(classificador, X_dados, y_alvo, cv=cv, scoring={
        "acuracia": "accuracy", "roc_auc": "roc_auc", "pr_auc": "average_precision",
        "recall": "recall", "precisao": "precision",
    })
    linha = {
        "abordagem": nome,
        "acuracia": resultado["test_acuracia"].mean(),
        "roc_auc": resultado["test_roc_auc"].mean(),
        "pr_auc": resultado["test_pr_auc"].mean(),
        "recall (classe rara)": resultado["test_recall"].mean(),
        "precisao (classe rara)": resultado["test_precisao"].mean(),
    }
    return linha


preguicoso = DummyClassifier(strategy="most_frequent")
linha_preguicoso = mede(preguicoso, X, y, "chuta sempre a classe majoritaria")
print("chutar sempre a classe majoritária:")
for chave, valor in linha_preguicoso.items():
    if chave != "abordagem":
        print(f"  {chave:24s} {valor:.4f}")
print()
print(f"recall da classe rara: {linha_preguicoso['recall (classe rara)']:.4f}, ou seja, "
      f"o modelo acha {linha_preguicoso['recall (classe rara)']:.1%} dos casos que importam")


Acurácia altíssima e revocação zero. O modelo acerta quase tudo porque quase tudo é "não", e não encontra
nenhum dos casos que o problema existe para encontrar.

Daqui em diante, três métricas mandam, e todas olham o evento raro: **PR-AUC** (a área sob a curva de
precisão-revocação, que mede o quão bem o modelo ordena os casos raros), **revocação** (quanto dos casos
raros ele pega) e **precisão** (de quanto ele aponta, quanto está certo). Acurácia fica na tabela só para
mostrar que ela se move pouco e engana.

## O ponto de partida, sem nenhuma alavanca

Um modelo de verdade, sem peso e sem reamostragem, na mesma validação cruzada.


In [ ]:
comum = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
linha_base = mede(comum, X, y, "sem alavanca nenhuma")
display(pd.DataFrame([linha_base]).set_index("abordagem").round(4))


Repare no que já apareceu: acurácia 0,9668, mais alta que a do modelo que chuta (0,9655), o que mostra que
ela não distingue nada; PR-AUC 0,3470 contra 0,0345 do chutador (que é exatamente a taxa de positivos); e
revocação de apenas 0,1082. O modelo **ordena bem** (a PR-AUC é dez vezes a taxa de positivos) e o corte
padrão de 0,5 é que o deixa quase inútil na prática.

## Alavanca 1: peso da classe

`class_weight="balanced"` multiplica o peso do erro na classe rara, proporcionalmente ao quanto ela é
rara. Não inventa linha, não joga linha fora, e não muda a tabela: muda a função que o treino minimiza.


In [ ]:
com_peso = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000,
                                                             class_weight="balanced"))
linha_peso = mede(com_peso, X, y, "class_weight=balanced")
display(pd.DataFrame([linha_base, linha_peso]).set_index("abordagem").round(4))


## Alavanca 2: reamostragem, e a regra que não pode ser quebrada

O SMOTE cria linhas sintéticas da minoria: pega um caso raro, sorteia um vizinho raro e inventa um ponto
entre os dois. A subamostragem faz o contrário: joga fora linhas da maioria até equilibrar.

A regra é uma só, e vale para as duas: **a reamostragem acontece no treino, depois da divisão**. Quem
garante isso em código é o `Pipeline` do `imbalanced-learn`, que reamostra dentro do ajuste de cada fold.


In [ ]:
com_smote = PipelineDesbalanceado([
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("escala", StandardScaler()),
    ("modelo", LogisticRegression(max_iter=5000)),
])
com_sub = PipelineDesbalanceado([
    ("subamostra", RandomUnderSampler(random_state=RANDOM_STATE)),
    ("escala", StandardScaler()),
    ("modelo", LogisticRegression(max_iter=5000)),
])

linha_smote = mede(com_smote, X, y, "SMOTE no pipeline")
linha_sub = mede(com_sub, X, y, "subamostragem no pipeline")
display(pd.DataFrame([linha_base, linha_peso, linha_smote, linha_sub])
        .set_index("abordagem").round(4))


A tabela conta uma história mais interessante que "balancear melhora": com peso, com SMOTE ou com
subamostragem, a **revocação** sobe de 0,1082 para 0,7746, 0,7749 e 0,7966, e a **PR-AUC cai** nas três:
0,3470 para 0,3246, 0,3240 e 0,2981. A acurácia também cai, de 0,9668 para cerca de 0,79, e isso não é
problema: ela estava alta justamente porque o modelo ignorava a classe rara.

Ou seja: as duas alavancas **trocam precisão por revocação**, não aumentam a qualidade da ordenação. Se o
seu problema é "achar o máximo de casos raros", elas resolvem. Se é "ordenar os casos por risco", o ganho
é pequeno.

## A armadilha: reamostrar antes de dividir

Se o SMOTE rodar na tabela inteira antes da validação, o modelo treina com linhas sintéticas geradas a
partir de casos que estão no teste, e o teste deixa de ser teste. Não gera erro, não gera aviso: gera
número.


In [ ]:
X_vazado, y_vazado = SMOTE(random_state=RANDOM_STATE).fit_resample(X, y)
print(f"tabela depois do SMOTE aplicado na base inteira: {len(X_vazado)} linhas, "
      f"{y_vazado.mean():.1%} de positivos")
print("agora as linhas sinteticas dos casos de teste ja estao no treino de cada fold")
print()
linha_vazado = mede(make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
                    X_vazado, y_vazado, "SMOTE antes da divisao (vazado)")
display(pd.DataFrame([linha_base, linha_smote, linha_vazado]).set_index("abordagem").round(4))


A PR-AUC vai de 0,3240, com o SMOTE dentro do pipeline, para **0,8806** com o SMOTE antes da divisão, e a
precisão sai de 0,1256 para 0,8075. As duas linhas usam SMOTE, o mesmo modelo e a mesma validação
cruzada: a única coisa que muda é **onde** a reamostragem acontece. Feito do jeito certo, o número é
honesto e não é bonito; feito do jeito errado, ele vira o melhor da tabela sem valer nada.

## Alavanca 3: o limiar de decisão

A última alavanca não mexe no dado nem no modelo: mexe no ponto onde a probabilidade vira resposta. O
modelo devolve uma nota de risco; o corte em 0,5 é uma convenção, e para um evento de 3% ela é alta
demais.

Esta alavanca é a mais barata das três, e a única que dá para ajustar sem treinar nada de novo.


In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.25, stratify=y,
                                                       random_state=RANDOM_STATE)
modelo_simples = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(X_treino, y_treino)
probabilidades = modelo_simples.predict_proba(X_teste)[:, 1]

limiares = [0.5, 0.3, 0.2, 0.1, 0.05, 0.03, 0.01]
linhas_limiar = []
for limiar in limiares:
    previsao = (probabilidades >= limiar).astype(int)
    linhas_limiar.append({
        "limiar": limiar,
        "acuracia": accuracy_score(y_teste, previsao),
        "recall (classe rara)": recall_score(y_teste, previsao),
        "precisao (classe rara)": precision_score(y_teste, previsao, zero_division=0),
        "f1": f1_score(y_teste, previsao),
    })

tabela_limiar = pd.DataFrame(linhas_limiar).set_index("limiar").round(4)
display(tabela_limiar)

precisao, recall, _ = precision_recall_curve(y_teste, probabilidades)
sns.lineplot(x=recall, y=precisao)
plt.axhline(y_teste.mean(), color="gray", linestyle="--", linewidth=1)
plt.title("curva de precisao-revocação, com a taxa de positivos na linha tracejada")
plt.xlabel("revocação")
plt.ylabel("precisão")
plt.tight_layout()
plt.show()


Baixando o limiar de 0,50 para 0,05, a revocação vai de 0,0882 para 0,6765 e a precisão cai de 0,7500
para 0,1565, sem treinar nada de novo. E note o que a tabela do resumo mostra: a PR-AUC do modelo simples
(0,3470) é maior que a de qualquer abordagem reamostrada. O que faltava não era dado equilibrado, era
ponto de operação.

Não existe limiar certo em abstrato: existe o limiar que corresponde ao custo do seu problema. Se deixar
passar um caso raro custa caro, o limiar desce; se cada alarme falso custa caro, o limiar sobe.

O `f1` da tabela é só um exemplo de critério de escolha, e não um bom critério em geral: ele trata
precisão e revocação como igualmente importantes, o que raramente é verdade.

## O resumo das alavancas

Juntando as abordagens numa tabela só, com a métrica que importa no começo e a que engana no fim.


In [ ]:
resumo = pd.DataFrame([linha_preguicoso, linha_base, linha_peso, linha_smote, linha_sub, linha_vazado])
resumo = resumo.set_index("abordagem")[["pr_auc", "recall (classe rara)", "precisao (classe rara)",
                                       "roc_auc", "acuracia"]]
display(resumo.round(4))
print("a linha vazada é a única que parece boa de verdade, e é a única que não vale")


## Como ler isso

1. se o evento é raro, **comece pela métrica**: acurácia fora, PR-AUC e revocação dentro;
2. antes de qualquer alavanca, saiba qual é a linha de base da PR-AUC: ela é a taxa de positivos, e tudo
   abaixo disso é modelo pior que o acaso;
3. `class_weight` é a alavanca mais barata e a menos invasiva: não mexe no dado;
4. SMOTE e subamostragem só **dentro** do pipeline de reamostragem, nunca antes da divisão;
5. limiar é a alavanca que não treina nada: ajuste-o pelo custo do erro, não por convenção;
6. desconfie de melhora grande em dado raro: quase sempre é vazamento de reamostragem ou de seleção.

## Resumo

Dado desbalanceado não é um problema de modelo, é um problema de métrica e de ponto de operação. Neste
notebook: um modelo que chuta a classe majoritária acerta 0,9655 e acha 0,0% dos casos que importam; o
`class_weight`, o SMOTE e a subamostragem levam a revocação de 0,1082 para cerca de 0,78, ao custo da
precisão (0,6267 para 0,1114 a 0,1256) e **sem melhorar a ordenação**, porque a PR-AUC cai de 0,3470
para 0,3246, 0,3240 e 0,2981; e baixar o limiar do modelo simples, sem reamostrar nada, leva a revocação
de 0,0882 a 0,6765.

E a armadilha, que é a linha mais perigosa da tabela: a mesma reamostragem aplicada antes da divisão
produz PR-AUC de 0,8806 e precisão de 0,8075, os melhores números do notebook, porque o teste deixou de
ser teste.

**Próximo passo:** a pasta `time-series`, que começa pelo protocolo de previsão, onde a divisão deixa de
ser aleatória e passa a ser temporal.

## Referências

```bibtex
@article{chawla2002smote,
  title   = {SMOTE: Synthetic Minority Over-Sampling Technique},
  author  = {Chawla, Nitesh V. and Bowyer, Kevin W. and Hall, Lawrence O. and Kegelmeyer, W. Philip},
  journal = {Journal of Artificial Intelligence Research},
  volume  = {16},
  pages   = {321--357},
  year    = {2002},
  doi     = {10.1613/jair.953}
}

@article{he2009learning,
  title   = {Learning from Imbalanced Data},
  author  = {He, Haibo and Garcia, Edwardo A.},
  journal = {IEEE Transactions on Knowledge and Data Engineering},
  volume  = {21},
  number  = {9},
  pages   = {1263--1284},
  year    = {2009},
  doi     = {10.1109/TKDE.2008.239}
}

@article{saito2015precision,
  title   = {The Precision-Recall Plot Is More Informative than the ROC Plot When Evaluating Binary
             Classifiers on Imbalanced Datasets},
  author  = {Saito, Takaya and Rehmsmeier, Marc},
  journal = {PLOS ONE},
  volume  = {10},
  number  = {3},
  pages   = {e0118432},
  year    = {2015},
  doi     = {10.1371/journal.pone.0118432}
}
```
